# Week 63, Day 4: Stochastic variational inference and the reparameterisation trick

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w063/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Variance of the two estimators

For q = N(m, 1) with m = 1 and f(z) = (z - 2) squared, compute 50,000 single-sample estimates of the gradient with respect to m by each method, and compare their means and standard deviations with the true gradient 2(m - 2).

In [ ]:
import numpy as np
ge = np.random.default_rng(634)
m = 1.0
f = lambda z: (z - 2) ** 2
df = lambda z: 2 * (z - 2)
eps = ge.normal(size=50000)
z = m + eps
g_score = ...                                          # f(z) times d/dm log N(z; m, 1)
g_rep = ...                                            # df/dz times dz/dm
true_grad = 2 * (m - 2)
print(f"true gradient {true_grad}; score function mean {g_score.mean():.3f} (sd {g_score.std():.2f}); reparameterised mean {g_rep.mean():.3f} (sd {g_rep.std():.2f})")

In [ ]:
import numpy as np
def test_unbiased():
    "both estimators average to the true gradient within 0.05"
    assert abs(np.mean(g_score) - true_grad) < 0.05 and abs(np.mean(g_rep) - true_grad) < 0.05
def test_variance():
    "the reparameterised estimator is at least twice as precise (smaller standard deviation)"
    assert np.std(g_score) > 2 * np.std(g_rep)

test_unbiased()
test_variance()
print("All checks passed")

### Exercise: Stochastic VI for logistic regression

Complete the update: draw 8 samples of w by reparameterisation, compute the gradient of the log joint for each, and take Adam-free gradient steps (learning rate 0.01) on m and log s. Compare the final mean with the MAP estimate found by Newton's method.

In [ ]:
gs = np.random.default_rng(6341)
n, p = 300, 6
X = gs.normal(size=(n, p))
w_true = np.array([1.5, -1.0, 0.5, 0.0, 0.0, 0.8])
y = (gs.random(n) < 1 / (1 + np.exp(-X @ w_true))).astype(float)
alpha = 1.0                                            # prior variance of each weight
sig = lambda u: 1 / (1 + np.exp(-u))
grad_logjoint = lambda w: X.T @ (y - sig(X @ w)) - w / alpha

w_map = np.zeros(p)                                     # Newton's method for the MAP
for _ in range(30):
    pr = sig(X @ w_map)
    H = X.T @ (X * (pr * (1 - pr))[:, None]) + np.eye(p) / alpha
    w_map += np.linalg.solve(H, grad_logjoint(w_map))

m_q, log_s = np.zeros(p), np.full(p, -1.0)
for it in range(3000):
    s = np.exp(log_s)
    eps = gs.normal(size=(8, p))
    G = np.array([grad_logjoint(m_q + s * e) for e in eps])
    m_q += 0.01 * G.mean(0)
    log_s += ...
s_q = np.exp(log_s)
print("MAP weights:        ", np.round(w_map, 3))
print("variational mean:   ", np.round(m_q, 3))
print("variational sd:     ", np.round(s_q, 3))
print("Laplace sd (for comparison):", np.round(np.sqrt(np.diag(np.linalg.inv(H))), 3))

In [ ]:
import numpy as np
def test_mean():
    "the variational mean is close to the MAP weights (within 0.15)"
    assert np.abs(m_q - w_map).max() < 0.15
def test_sd():
    "the variational standard deviations are positive and smaller than the prior's"
    assert np.all(s_q > 0) and np.all(s_q < 1)

test_mean()
test_sd()
print("All checks passed")

### Exercise: An amortised encoder by reparameterised gradients

Complete the gradient of the minibatch ELBO with respect to the encoder matrix A: the reconstruction term gives (W^T (x - W z)) / s2 times x^T, and the KL term gives minus the mean (A x) times x^T, both averaged over the minibatch. The gradient for log s is given.

In [ ]:
gv = np.random.default_rng(6342)
d, k, s2 = 12, 2, 0.5
W = gv.normal(size=(d, k))
M = W.T @ W + s2 * np.eye(k)
Xd = gv.normal(size=(20000, k)) @ W.T + np.sqrt(s2) * gv.normal(size=(20000, d))   # centred data

A = np.zeros((k, d)); log_s = np.zeros(k)
lr = 0.002
for it in range(8000):
    xb = Xd[gv.integers(0, len(Xd), 64)]                  # minibatch, 64 x d
    s = np.exp(log_s)
    eps = gv.normal(size=(64, k))
    mean = xb @ A.T
    zb = mean + s * eps                                   # reparameterised samples
    resid = xb - zb @ W.T                                  # 64 x d
    gz = resid @ W / s2                                    # d(reconstruction)/dz, 64 x k
    grad_A = ...                                          # reconstruction term plus KL term, averaged
    grad_log_s = (gz * eps).mean(0) * s - (s ** 2 - 1)     # reconstruction through z, and KL(q || N(0, I))
    A += lr * grad_A
    log_s += lr * grad_log_s
print("largest difference from the exact encoder:", np.abs(A - np.linalg.solve(M, W.T)).max().round(4))
print("learned sd:", np.round(np.exp(log_s), 4), " mean-field optimum:", np.round(np.sqrt(s2 / np.diag(M)), 4))

In [ ]:
import numpy as np
def test_mean_map():
    "the learned encoder matrix is close to the exact posterior-mean matrix"
    assert np.abs(A - np.linalg.solve(M, W.T)).max() < 0.05
def test_var():
    "the learned standard deviations match the mean-field optimum to within 10%"
    assert np.allclose(np.exp(log_s), np.sqrt(s2 / np.diag(M)), rtol=0.1)

test_mean_map()
test_var()
print("All checks passed")